In [1]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import gaussian_kde
from scipy.spatial.distance import euclidean
from scipy.interpolate import griddata
import seaborn as sns
from PIL import Image
import glob
import os
import pickle
from typing import List, Tuple
import json


# Avalanche acquistion

This notebook 1) opens EEG data; 2) interpolate the missing electrodes; 3)

## Open EEG and interpolate

In [2]:
#!/usr/bin/env python3
"""
Fixed interpolation that maintains proper channel order alignment with electrode positions.
"""
def interpolate_missing_electrodes(raw_or_epochs, positions_file='electrode_positions.json', 
                                 force_interpolate=None, verbose=True):
    """
    Interpolate missing electrodes using custom positions with proper channel ordering.
    
    Parameters
    ----------
    raw_or_epochs : mne.Raw or mne.Epochs
        Input EEG data
    positions_file : str
        Path to JSON file with electrode positions
    force_interpolate : list or None
        Channels to force re-interpolation
    verbose : bool
        Print progress
    
    Returns
    -------
    data_interpolated : mne.Raw or mne.Epochs
        Data with interpolated channels in correct order
    interpolated_channels : list
        List of interpolated channels
    """
    
    if force_interpolate is None:
        force_interpolate = []
    
    data_copy = raw_or_epochs.copy()
    
    # Load positions
    with open(positions_file, 'r') as f:
        electrode_data = json.load(f)
    
    ch_pos = {}
    for ch, pos_dict in electrode_data.items():
        ch_pos[ch] = np.array([pos_dict['x'], pos_dict['y'], pos_dict['z']])
    
    # Get the expected channel order from JSON (this is the target order)
    expected_channel_order = list(electrode_data.keys())
    
    # Rename FP1 -> Fp1 if needed
    if 'FP1' in data_copy.ch_names and 'FP1' not in ch_pos and 'Fp1' in ch_pos:
        data_copy.rename_channels({'FP1': 'Fp1'})
        if verbose:
            print("Renamed FP1 -> Fp1")
    
    # Find channels to interpolate
    current = set(data_copy.ch_names)
    available = set(ch_pos.keys())
    missing = list(available - current)
    forced = [ch for ch in force_interpolate if ch in current]
    to_interpolate = list(set(missing + forced))
    
    if not to_interpolate:
        # Even if no interpolation needed, ensure correct channel order
        try:
            # Only reorder channels that exist in both data and expected order
            existing_in_expected = [ch for ch in expected_channel_order if ch in data_copy.ch_names]
            data_copy = data_copy.reorder_channels(existing_in_expected)
            if verbose:
                print("Reordered existing channels to match electrode positions")
        except:
            if verbose:
                print("Could not reorder channels - some channels may not match electrode positions")
        return data_copy, []
    
    if verbose:
        print(f"Interpolating {len(to_interpolate)} channels: {to_interpolate}")
    
    # Source channels for interpolation
    sources = {ch: ch_pos[ch] for ch in data_copy.ch_names 
               if ch not in to_interpolate and ch in ch_pos}
    
    # Store original data before removing channels
    original_data = data_copy.get_data()
    original_ch_names = data_copy.ch_names.copy()
    ch_to_idx = {ch: idx for idx, ch in enumerate(original_ch_names)}
    
    # Remove existing channels that will be re-interpolated
    existing_to_remove = [ch for ch in to_interpolate if ch in data_copy.ch_names]
    if existing_to_remove:
        data_copy = data_copy.pick_channels([ch for ch in data_copy.ch_names if ch not in existing_to_remove])
    
    # Interpolate each channel
    interpolated_data = []
    successful = []
    
    for ch in to_interpolate:
        if ch not in ch_pos:
            continue
        
        # Find 6 nearest neighbors
        target_pos = ch_pos[ch]
        distances = [(name, np.linalg.norm(target_pos - pos)) for name, pos in sources.items()]
        neighbors = sorted(distances, key=lambda x: x[1])[:6]
        
        if not neighbors:
            if verbose:
                print(f"  {ch}: No neighbors found, skipping")
            continue
        
        # Weights by inverse squared distance
        weights = np.array([1/(d**2) if d > 1e-10 else 1.0 for _, d in neighbors])
        weights /= weights.sum()
        
        # Weighted average
        neighbor_idx = [ch_to_idx[name] for name, _ in neighbors if name in ch_to_idx]
        if not neighbor_idx:
            if verbose:
                print(f"  {ch}: No valid neighbor indices found, skipping")
            continue
            
        if original_data.ndim == 2:  # Raw
            signal = sum(w * original_data[idx] for w, idx in zip(weights[:len(neighbor_idx)], neighbor_idx))
        else:  # Epochs
            signal = sum(w * original_data[:, idx, :] for w, idx in zip(weights[:len(neighbor_idx)], neighbor_idx))
        
        interpolated_data.append(signal)
        successful.append(ch)
        
        if verbose:
            neighbor_names = [name for name, _ in neighbors[:len(neighbor_idx)]]
            print(f"  {ch}: from {neighbor_names}")
    
    # Add interpolated channels back
    if successful:
        if original_data.ndim == 2:
            interp_array = np.array(interpolated_data)
        else:
            interp_array = np.transpose(np.array(interpolated_data), (1, 0, 2))
        
        interp_info = mne.create_info(successful, data_copy.info['sfreq'], ['eeg'] * len(successful))
        
        if original_data.ndim == 2:
            interp_obj = mne.io.RawArray(interp_array, interp_info, verbose=False)
        else:
            interp_obj = mne.EpochsArray(interp_array, interp_info, verbose=False)
        
        data_copy = data_copy.add_channels([interp_obj], force_update_info=True)
    
    # CRITICAL FIX: Reorder channels to match electrode positions JSON
    try:
        # Only include channels that exist in the final data
        final_channel_order = [ch for ch in expected_channel_order if ch in data_copy.ch_names]
        
        if verbose:
            print(f"Original order: {data_copy.ch_names[:10]}...")
            print(f"Target order:   {final_channel_order[:10]}...")
        
        # Reorder to match electrode positions
        data_copy = data_copy.reorder_channels(final_channel_order)
        
        if verbose:
            print(f"✓ Reordered {len(data_copy.ch_names)} channels to match electrode positions")
            print(f"Final order:    {data_copy.ch_names[:10]}...")
            
    except Exception as e:
        if verbose:
            print(f"Warning: Could not reorder channels: {e}")
            print("Channel order may not match electrode positions!")
    
    if verbose:
        print(f"✓ Interpolated {len(successful)} channels")
        print(f"✓ Final data has {len(data_copy.ch_names)} channels in correct order")
    
    return data_copy, successful


# Example usage function to test the fix
def test_channel_ordering(raw_or_epochs, positions_file):
    """Test function to verify channel ordering is correct."""
    
    # Load electrode positions to get expected order
    with open(positions_file, 'r') as f:
        electrode_data = json.load(f)
    expected_order = list(electrode_data.keys())
    
    print("=== BEFORE INTERPOLATION ===")
    print(f"Expected order (first 10): {expected_order[:10]}")
    print(f"Data order (first 10):     {raw_or_epochs.ch_names[:10]}")
    print(f"Orders match: {expected_order[:len(raw_or_epochs.ch_names)] == raw_or_epochs.ch_names}")
    
    # Apply fixed interpolation
    data_fixed, interpolated = interpolate_missing_electrodes(raw_or_epochs, positions_file)
    
    print("\n=== AFTER INTERPOLATION ===")
    print(f"Expected order (first 10): {expected_order[:10]}")
    print(f"Data order (first 10):     {data_fixed.ch_names[:10]}")
    final_expected = [ch for ch in expected_order if ch in data_fixed.ch_names]
    print(f"Orders match: {final_expected == data_fixed.ch_names}")
    
    return data_fixed, interpolated

In [ ]:
# Load and interpolate .set files
# Define base directories and conditions
base_dirs = [
    '/home/novicky/psiconnect/Control',
    '/home/novicky/psiconnect/Psilocybin'
]

conditions = ['Meditation', 'Music', 'Resting state', 'Video']

# Collect all .set files
file_paths = []

for base_dir in base_dirs:
    for condition in conditions:
        # Create the full path pattern
        pattern = os.path.join(base_dir, condition, '*.set')
        
        # Find all .set files in this directory
        files = glob.glob(pattern)
        
        # Add to the list
        file_paths.extend(files)

# Sort the files for consistent ordering
file_paths.sort()

print(f"Found {len(file_paths)} .set files:")
for path in file_paths:
    print(path)
datasets = []

for file_path in file_paths:
    raw = mne.io.read_raw_eeglab(file_path, preload=True)
    raw.pick_types(eeg=True, exclude='bads')
    
    # Interpolate missing electrodes right here
    raw_interp, interp_channels = interpolate_missing_electrodes(
        raw, 
        positions_file='/home/novicky/clean_code/electrode_positions.json'
    )
    # Downsample from 500 Hz to 250 Hz
    raw_interp.resample(250, npad='auto')

    # Now extract data - all datasets will have same channels
    data = raw_interp.get_data()  # (channels x samples)
    datasets.append(data)

## Next, get the avalanches above and below threshold

In [4]:
def create_avalanche_filter(datasets, crossover_thresholds):
    """
    Create filter masks based on crossover thresholds for multiple datasets.
    
    Parameters:
    -----------
    datasets : list of arrays
        List of EEG datasets, each with shape (n_channels, n_samples)
    crossover_thresholds : list
        Crossover points from analyze_crossovers
    
    Returns:
    --------
    filter_masks : list of arrays
        List of boolean masks for each dataset, each with shape (n_samples,)
        indicating which time points have activity exceeding thresholds
    """
    
    filter_masks = []
    
    # Get thresholds
    min_threshold = min(crossover_thresholds)
    max_threshold = max(crossover_thresholds)
    
    for i, dataset in enumerate(datasets):
        dataset = np.asarray(dataset)
        
        # Handle different data shapes
        if dataset.ndim == 1:
            dataset = dataset.reshape(1, -1)
        elif dataset.ndim == 2:
            if dataset.shape[0] > dataset.shape[1]:
                dataset = dataset.T
        
        # Normalize data (same as in crossover analysis) - only for threshold detection
        means = np.mean(dataset, axis=1, keepdims=True)
        stds = np.std(dataset, axis=1, keepdims=True)
        stds[stds == 0] = 1
        normalized_data = (dataset - means) / stds
        
        # Create filter mask: any electrode exceeding thresholds at each time point
        filter_mask = np.any((normalized_data <= min_threshold) | (normalized_data >= max_threshold), axis=0)
        
        filter_masks.append(filter_mask)
    
    return filter_masks


def detect_avalanches(datasets, filter_masks, crossover_thresholds, min_duration=2, verbose=True):
    """
    Detect avalanches by tracking time indices and including all electrodes for multiple datasets.
    
    Parameters:
    -----------
    datasets : list of arrays
        List of original EEG datasets (real values, not normalized), each with shape (n_channels, n_samples)
    filter_masks : list of arrays
        List of boolean masks indicating avalanche time points for each dataset
    min_duration : int
        Minimum duration for valid avalanches
    crossover_thresholds : list
        Crossover points for avalanche detection
    verbose : bool
        Print detection details
    
    Returns:
    --------
    all_avalanches : list of lists
        List containing avalanche lists for each dataset. Each inner list contains
        avalanche matrices with original data values, each with shape (n_electrodes, duration)
    all_avalanche_times : list of lists
        List containing avalanche timing lists for each dataset. Each inner list contains
        tuples of (start_idx, end_idx) indicating when each avalanche occurs in the original dataset
    """
    
    if len(datasets) != len(filter_masks):
        raise ValueError(f"Number of datasets ({len(datasets)}) must match number of filter masks ({len(filter_masks)})")
    
    all_avalanches = []
    all_avalanche_times = []
    
    if verbose:
        print(f"Detecting avalanches for {len(datasets)} datasets...")

    min_threshold = min(crossover_thresholds)
    max_threshold = max(crossover_thresholds)
    
    for dataset_idx, (dataset, filter_mask) in enumerate(zip(datasets, filter_masks)):
        dataset = np.asarray(dataset)
        
        # Handle different data shapes
        if dataset.ndim == 1:
            dataset = dataset.reshape(1, -1)
        elif dataset.ndim == 2:
            if dataset.shape[0] > dataset.shape[1]:
                dataset = dataset.T
        
        n_electrodes, n_samples = dataset.shape
        
        if np.sum(filter_mask) == 0:
            if verbose:
                print(f"Dataset {dataset_idx + 1}: Warning: No data samples available for avalanche detection")
            
            all_avalanches.append([])
            all_avalanche_times.append([])
            continue
        
        if verbose:
            print(f"\nDataset {dataset_idx + 1}:")
            print(f"  Data shape: {dataset.shape} (n_electrodes, n_samples)")
            print(f"  Active time points: {np.sum(filter_mask)} ({np.sum(filter_mask)/n_samples*100:.2f}%)")
        
        # Find runs of consecutive True values in the filter_mask
        padded_mask = np.concatenate(([False], filter_mask, [False]))
        diff = np.diff(padded_mask.astype(int))
        
        # Find start and end points of continuous activity
        run_starts = np.where(diff == 1)[0]
        run_ends = np.where(diff == -1)[0]
        
        if verbose:
            print(f"  Found {len(run_starts)} continuous runs")
        
        avalanches = []
        avalanche_times = []
        
        # Process each run
        for start_idx, end_idx in zip(run_starts, run_ends):
            duration = end_idx - start_idx
            
            if duration >= min_duration:
                # Extract avalanche data for ALL electrodes during this time period (original values)
                # Create electrode-specific mask for this time period
                time_slice = dataset[:, start_idx:end_idx]
                means = np.mean(dataset, axis=1, keepdims=True)
                stds = np.std(dataset, axis=1, keepdims=True)
                stds[stds == 0] = 1
                normalized_slice = (time_slice - means) / stds

                # Mask for electrodes exceeding thresholds
                electrode_mask = (normalized_slice <= min_threshold) | (normalized_slice >= max_threshold)

                # Zero out non-thresholded data points
                avalanche_matrix = time_slice.copy()
                avalanche_matrix[~electrode_mask] = 0
                
                avalanches.append(avalanche_matrix)
                avalanche_times.append((start_idx, end_idx))
                
                if verbose and len(run_starts) <= 20:
                    print(f"    Avalanche: time [{start_idx}:{end_idx}], duration={duration}, shape={avalanche_matrix.shape}")
        
        if verbose:
            print(f"  Avalanche Detection Results:")
            print(f"    Total avalanches detected: {len(run_starts)}")
            print(f"    Avalanches kept (≥{min_duration} steps): {len(avalanches)}")
            if len(avalanches) > 0:
                durations = [end - start for start, end in avalanche_times]
                print(f"    Duration range: {min(durations)} - {max(durations)} time steps")
                print(f"    Each avalanche contains original data values for all {n_electrodes} electrodes")
        
        all_avalanches.append(avalanches)
        all_avalanche_times.append(avalanche_times)
    
    return all_avalanches, all_avalanche_times

In [ ]:
# Create the filter mask using the dataset and crossover thresholds

average_crossovers = [-3,3]
print(f"\nUsing average crossovers: {average_crossovers}")

filter_mask = create_avalanche_filter(datasets, average_crossovers)

# Detect avalanches using the original dataset and the filter mask
all_avalanches, all_avalanche_times = detect_avalanches(datasets, filter_mask, average_crossovers,min_duration=11)


Using average crossovers: [-3, 3]


Detecting avalanches for 429 datasets...

Dataset 1:
  Data shape: (64, 74420) (n_electrodes, n_samples)
  Active time points: 5323 (7.15%)
  Found 1260 continuous runs
  Avalanche Detection Results:
    Total avalanches detected: 1260
    Avalanches kept (≥1 steps): 1260
    Duration range: 1 - 34 time steps
    Each avalanche contains original data values for all 64 electrodes

Dataset 2:
  Data shape: (64, 73826) (n_electrodes, n_samples)
  Active time points: 3112 (4.22%)
  Found 890 continuous runs
  Avalanche Detection Results:
    Total avalanches detected: 890
    Avalanches kept (≥1 steps): 890
    Duration range: 1 - 24 time steps
    Each avalanche contains original data values for all 64 electrodes

Dataset 3:
  Data shape: (64, 56612) (n_electrodes, n_samples)
  Active time points: 6659 (11.76%)
  Found 1455 continuous runs
  Avalanche Detection Results:
    Total avalanches detected: 1455
    Avalanches kept (≥1 steps): 1455
    Duration range: 1 - 203 time steps
    Each

### Get the distances among EEG electrodes


In [6]:
"""
Create EEG ground metric using custom electrode positions from JSON.
"""

def create_eeg_ground_metric(positions_file='electrode_positions.json'):
    """
    Create ground metric based on actual EEG electrode positions from JSON file.
    
    Parameters:
    -----------
    positions_file : str
        Path to JSON file containing electrode positions
    
    Returns:
    --------
    K : numpy.ndarray
        Ground metric matrix based on electrode distances
    channel_names : list
        List of channel names used
    """
    
    print(f"Creating EEG ground metric from {positions_file}...")
    
    # Load electrode positions from JSON
    with open(positions_file, 'r') as f:
        electrode_data = json.load(f)
    
    # Extract channel names and positions
    channel_names = list(electrode_data.keys())
    n_channels = len(channel_names)
    
    print(f"Loaded {n_channels} channels: {channel_names}")
    
    # Convert positions to numpy array
    positions_3d = []
    for ch_name in channel_names:
        pos_dict = electrode_data[ch_name]
        pos = np.array([pos_dict['x'], pos_dict['y'], pos_dict['z']])
        positions_3d.append(pos)
    
    positions_3d = np.array(positions_3d)
    print(f"Using {len(positions_3d)} electrode positions")
    
    # Compute pairwise Euclidean distances
    K = np.zeros((n_channels, n_channels))
    
    for i in range(n_channels):
        for j in range(n_channels):
            if i == j:
                K[i, j] = 0.0
            else:
                # Euclidean distance in 3D space
                dist = np.linalg.norm(positions_3d[i] - positions_3d[j])
                K[i, j] = dist
    
    print(f"Ground metric statistics:")
    print(f"  Shape: {K.shape}")
    print(f"  Min distance: {K.min():.4f}")
    print(f"  Max distance: {K.max():.4f}")
    print(f"  Median distance: {np.median(K[K > 0]):.4f}")
    
    return K, channel_names

In [7]:

K, channel_names = create_eeg_ground_metric('/home/novicky/clean_code/electrode_positions.json')
print(f"Ground metric K shape: {K.shape}")

Creating EEG ground metric from /home/novicky/clean_code/electrode_positions.json...
Loaded 64 channels: ['Fp1', 'Fz', 'F3', 'F7', 'FT9', 'FC5', 'FC1', 'C3', 'T7', 'TP9', 'CP5', 'CP1', 'Pz', 'P3', 'P7', 'O1', 'Oz', 'O2', 'P4', 'P8', 'TP10', 'CP6', 'CP2', 'Cz', 'C4', 'T8', 'FT10', 'FC6', 'FC2', 'F4', 'F8', 'Fp2', 'AF7', 'AF3', 'AFz', 'F1', 'F5', 'FT7', 'FC3', 'C1', 'C5', 'TP7', 'CP3', 'P1', 'P5', 'PO7', 'PO3', 'POz', 'PO4', 'PO8', 'P6', 'P2', 'CPz', 'CP4', 'TP8', 'C6', 'C2', 'FC4', 'FT8', 'F6', 'AF8', 'AF4', 'F2', 'Iz']
Using 64 electrode positions
Ground metric statistics:
  Shape: (64, 64)
  Min distance: 0.0000
  Max distance: 0.2000
  Median distance: 0.1330
Ground metric K shape: (64, 64)


Consider adding the below instead of above for requiring both positive and negative bursts of activity

In [8]:
import numpy as np
import json
from scipy.spatial import ConvexHull

def filter_avalanche_propagation(all_avalanches, all_avalanche_times, K, electrode_positions_file, 
                                  min_active_neighbors=5, n_nearest=8):
    """
    Filter avalanches to keep only those showing dense spatial clustering in BOTH
    negative and positive activations, excluding boundary electrodes.
    
    Parameters:
    -----------
    all_avalanches : list of lists
        List containing avalanche lists for each participant. Each inner list contains
        avalanche matrices, each with shape (n_electrodes, duration)
    all_avalanche_times : list of lists
        List containing avalanche timing lists for each participant. Each inner list contains
        tuples of (start_idx, end_idx) indicating when each avalanche occurs
    K : np.ndarray
        Pre-computed distance matrix between electrodes (n_electrodes, n_electrodes)
    electrode_positions_file : str
        Path to JSON file with electrode positions (used for boundary detection)
    min_active_neighbors : int
        Minimum number of active neighbors required (default: 5)
    n_nearest : int
        Number of nearest neighbors to consider (default: 8)
    
    Returns:
    --------
    filtered_avalanches : list of lists
        Same structure as all_avalanches but containing only avalanches with spatial 
        clustering in both polarities
    filtered_avalanche_times : list of lists
        Same structure as all_avalanche_times but containing only timing info for 
        avalanches with spatial clustering in both polarities
    """
    
    if len(all_avalanches) != len(all_avalanche_times):
        raise ValueError(f"Number of avalanche lists ({len(all_avalanches)}) must match "
                        f"number of timing lists ({len(all_avalanche_times)})")
    
    # Load electrode positions
    with open(electrode_positions_file, 'r') as f:
        electrode_data = json.load(f)
    
    # Get excluded (boundary) electrodes
    excluded_electrodes = get_boundary_electrodes_with_occipital_exception(electrode_data)
    
    # Get channel names in order and convert excluded electrodes to indices
    all_channels = list(electrode_data.keys())
    excluded_indices = {i for i, ch in enumerate(all_channels) if ch in excluded_electrodes}
    included_channels = [ch for ch in all_channels if ch not in excluded_electrodes]
    
    print(f"Total electrodes: {len(all_channels)}")
    print(f"Excluded boundary electrodes: {len(excluded_electrodes)}")
    print(f"Excluded electrodes: {sorted(excluded_electrodes)}")
    print(f"Included electrodes for clustering: {len(included_channels)}")
    
    # Pre-compute nearest neighbors using K matrix (fast)
    nearest_neighbors = get_nearest_neighbors(K, n_nearest)
    
    filtered_avalanches = []
    filtered_avalanche_times = []
    
    for participant_idx, (participant_avalanches, participant_times) in enumerate(
            zip(all_avalanches, all_avalanche_times)):
        
        if len(participant_avalanches) != len(participant_times):
            raise ValueError(f"Participant {participant_idx + 1}: Number of avalanches "
                           f"({len(participant_avalanches)}) must match number of timing entries "
                           f"({len(participant_times)})")
        
        kept_avalanches = []
        kept_times = []
        
        print(f"\nProcessing participant {participant_idx + 1}: {len(participant_avalanches)} avalanches")
        
        for avalanche_idx, (avalanche, avalanche_time) in enumerate(
                zip(participant_avalanches, participant_times)):
            has_neg, has_pos = has_spatial_clustering_both_polarities(
                avalanche, excluded_indices, nearest_neighbors, min_active_neighbors
            )
            
            if has_neg and has_pos:
                kept_avalanches.append(avalanche)
                kept_times.append(avalanche_time)
        
        print(f"  Kept {len(kept_avalanches)}/{len(participant_avalanches)} avalanches "
              f"(both polarities with clustering)")
        
        # Always append the lists (even if empty) to maintain structure
        filtered_avalanches.append(kept_avalanches)
        filtered_avalanche_times.append(kept_times)
    
    return filtered_avalanches, filtered_avalanche_times

def get_boundary_electrodes_with_occipital_exception(electrode_data):
    """
    Find boundary electrodes using convex hull, but exclude occipital electrodes 
    from boundary removal (treat them as internal electrodes).
    
    Parameters:
    -----------
    electrode_data : dict
        Dictionary with electrode names as keys and position dicts as values
    
    Returns:
    --------
    set
        Set of electrode names that are on the boundary (excluding occipitals)
    """
    
    # Extract electrode names and 2D positions (x, y)
    electrode_names = list(electrode_data.keys())
    positions_2d = []
    
    for ch in electrode_names:
        pos = electrode_data[ch]
        positions_2d.append([pos['x'], pos['y']])
    
    positions_2d = np.array(positions_2d)
    
    # Compute convex hull
    try:
        hull = ConvexHull(positions_2d)
        boundary_indices = set(hull.vertices)
    except:
        print("Warning: Could not compute convex hull, using no boundary exclusion")
        return set()
    
    # Get electrode names on the boundary
    boundary_electrodes = {electrode_names[i] for i in boundary_indices}
    
    # Special handling: remove occipital electrodes from boundary exclusion
    occipital_electrodes = {'O1', 'O2', 'Oz', 'PO7', 'PO8'}
    
    # Find which occipitals are actually in the data
    occipitals_in_data = occipital_electrodes.intersection(set(electrode_names))
    
    # Remove occipitals from boundary exclusion (put them back in analysis)
    excluded_electrodes = boundary_electrodes - occipitals_in_data
    
    print(f"Convex hull boundary electrodes: {sorted(boundary_electrodes)}")
    print(f"Occipital electrodes found: {sorted(occipitals_in_data)}")
    print(f"Occipitals on boundary (kept in analysis): {sorted(boundary_electrodes.intersection(occipitals_in_data))}")
    
    return excluded_electrodes


def has_spatial_clustering_both_polarities(avalanche, excluded_indices, nearest_neighbors, min_active_neighbors=5):
    """
    Check if avalanche contains dense spatial clustering in BOTH negative and positive values,
    considering only non-boundary electrodes as seeds.
    
    Parameters:
    -----------
    avalanche : np.ndarray
        Avalanche matrix with shape (n_electrodes, duration)
    excluded_indices : set
        Set of electrode indices to exclude from clustering analysis
    nearest_neighbors : dict
        Pre-computed nearest neighbors mapping (from get_nearest_neighbors)
    min_active_neighbors : int
        Minimum number of active neighbors required
    
    Returns:
    --------
    tuple (bool, bool)
        (has_negative_clustering, has_positive_clustering)
    """
    
    n_electrodes, duration = avalanche.shape
    
    # Get indices of electrodes that can be used as seeds (non-excluded)
    seed_electrode_indices = [i for i in range(n_electrodes) if i not in excluded_indices]
    
    has_negative_clustering = False
    has_positive_clustering = False
    
    # Check each timestep
    for t in range(duration):
        timestep_data = avalanche[:, t]
        
        # Get negative and positive active electrodes separately
        negative_electrodes = np.where(timestep_data < 0)[0]
        positive_electrodes = np.where(timestep_data > 0)[0]
        
        # Check negative clustering (only from seed electrodes)
        if not has_negative_clustering and len(negative_electrodes) > 0:
            negative_set = set(negative_electrodes)
            seed_negative_electrodes = [idx for idx in negative_electrodes if idx in seed_electrode_indices]
            
            for electrode_idx in seed_negative_electrodes:
                neighbors = nearest_neighbors[electrode_idx]
                # Filter out excluded electrodes from neighbor consideration
                valid_neighbors = [n for n in neighbors if n not in excluded_indices]
                active_neighbor_count = sum(1 for neighbor in valid_neighbors if neighbor in negative_set)
                
                if active_neighbor_count >= min_active_neighbors:
                    has_negative_clustering = True
                    break
        
        # Check positive clustering (only from seed electrodes)
        if not has_positive_clustering and len(positive_electrodes) > 0:
            positive_set = set(positive_electrodes)
            seed_positive_electrodes = [idx for idx in positive_electrodes if idx in seed_electrode_indices]
            
            for electrode_idx in seed_positive_electrodes:
                neighbors = nearest_neighbors[electrode_idx]
                # Filter out excluded electrodes from neighbor consideration
                valid_neighbors = [n for n in neighbors if n not in excluded_indices]
                active_neighbor_count = sum(1 for neighbor in valid_neighbors if neighbor in positive_set)
                
                if active_neighbor_count >= min_active_neighbors:
                    has_positive_clustering = True
                    break
        
        # Early exit if both conditions met
        if has_negative_clustering and has_positive_clustering:
            return True, True
    
    return has_negative_clustering, has_positive_clustering


def get_nearest_neighbors(K, n_nearest):
    """
    Find the n nearest neighbors for each electrode based on distance matrix.
    
    Parameters:
    -----------
    K : np.ndarray
        Distance matrix between electrodes (n_electrodes, n_electrodes)
    n_nearest : int
        Number of nearest neighbors to find
    
    Returns:
    --------
    dict
        Dictionary mapping electrode_idx -> list of nearest neighbor indices
    """
    
    n_electrodes = K.shape[0]
    nearest_neighbors = {}
    
    for electrode_idx in range(n_electrodes):
        # Get distances from this electrode to all others
        distances = K[electrode_idx, :].copy()
        
        # Set self-distance to infinity to exclude it
        distances[electrode_idx] = np.inf
        
        # Find indices of n_nearest closest electrodes
        # argsort gives indices sorted by distance (ascending)
        sorted_indices = np.argsort(distances)
        neighbors = sorted_indices[:n_nearest].tolist()
        
        nearest_neighbors[electrode_idx] = neighbors
    
    return nearest_neighbors

In [ ]:
# Apply spatial clustering filter to avalanches
# Keeps only avalanches with 6+ active electrodes among 8 nearest neighbors

filtered_avalanches, filtered_avalanche_times = filter_avalanche_propagation(
    all_avalanches,
    all_avalanche_times, 
    K,
    electrode_positions_file='/home/novicky/clean_code/electrode_positions.json',
    min_active_neighbors=5,
    n_nearest=8
)

# Check results
print("\n" + "="*60)
print("FILTERING SUMMARY")
print("="*60)

total_before = sum(len(participant_avs) for participant_avs in all_avalanches)
total_after = sum(len(participant_avs) for participant_avs in filtered_avalanches)

print(f"\nTotal avalanches before filtering: {total_before}")
print(f"Total avalanches after filtering: {total_after}")
print(f"Kept: {total_after/total_before*100:.1f}%")

print("\nPer-participant breakdown:")
for i, (before, after) in enumerate(zip(all_avalanches, filtered_avalanches)):
    if len(before) > 0:
        kept_pct = len(after)/len(before)*100
        print(f"  Participant {i+1}: {len(after)}/{len(before)} ({kept_pct:.1f}%)")
    else:
        print(f"  Participant {i+1}: 0/0 (no avalanches)")

# Now use filtered_all_avalanches for further analysis
# It has the same structure as all_avalanches but with fewer avalanches

### Change the structure so it is compatible with STA

### Convert to binary peaks

In [10]:
def convert_to_peaks(filtered_data: list) -> list:
    """
    Convert existing avalanches to peak representation for all participants.
    
    Parameters:
    -----------
    filtered_data : list of lists
        List where each element is a participant, and each participant is a list 
        of avalanches. Each avalanche is a 2D array with shape (electrodes, duration).
    
    Returns:
    --------
    peak_avalanches : list of lists
        Same structure as input, but with peak representation where:
        - Positive values become 1
        - Negative values become -1
        - Zero values remain 0
    """
    peak_avalanches = []
    
    # Process each participant
    for participant_idx, participant_avalanches in enumerate(filtered_data):
        participant_peaks = []
        
        # Process each avalanche for this participant
        for avalanche_idx, avalanche in enumerate(participant_avalanches):
            # Convert to peak representation using np.sign
            peak_avalanche = np.sign(avalanche).astype(avalanche.dtype)
            participant_peaks.append(peak_avalanche)
        
        peak_avalanches.append(participant_peaks)
        print(f"Processed participant {participant_idx}: {len(participant_avalanches)} avalanches")
    
    return peak_avalanches

In [11]:
filtered_avalanche_peaks = convert_to_peaks(all_avalanches)

Processed participant 0: 1260 avalanches
Processed participant 1: 890 avalanches
Processed participant 2: 1455 avalanches
Processed participant 3: 1043 avalanches
Processed participant 4: 1377 avalanches
Processed participant 5: 1772 avalanches
Processed participant 6: 1377 avalanches
Processed participant 7: 1341 avalanches
Processed participant 8: 2419 avalanches
Processed participant 9: 1407 avalanches
Processed participant 10: 1511 avalanches
Processed participant 11: 1468 avalanches
Processed participant 12: 1530 avalanches
Processed participant 13: 1883 avalanches
Processed participant 14: 1682 avalanches
Processed participant 15: 2658 avalanches
Processed participant 16: 1791 avalanches
Processed participant 17: 1800 avalanches
Processed participant 18: 1787 avalanches
Processed participant 19: 1646 avalanches
Processed participant 20: 1451 avalanches
Processed participant 21: 1666 avalanches
Processed participant 22: 1551 avalanches
Processed participant 23: 1947 avalanches
Pro

In [ ]:
# Create the directory if it doesn't exist
output_dir = 'processed_avalanches_250Hz'
os.makedirs(output_dir, exist_ok=True)

# Save as pickle files
with open(os.path.join(output_dir, 'filtered_avalanches_250hz.pkl'), 'wb') as f:
    pickle.dump(all_avalanches, f)

with open(os.path.join(output_dir, 'filtered_avalanche_peaks_250hz.pkl'), 'wb') as f:
    pickle.dump(filtered_avalanche_peaks, f)

with open(os.path.join(output_dir, 'filtered_avalanche_times_250hz.pkl'), 'wb') as f:
    pickle.dump(all_avalanche_times, f)

print(f"All files saved in '{output_dir}':")
print("- filtered_avalanches_250hz.pkl")
print("- filtered_avalanche_peaks_250hz.pkl")
print("- filtered_avalanche_times_250hz.pkl")

All files saved in 'processed_avalanches_full':
- filtered_avalanches_full.pkl
- filtered_avalanche_peaks_full.pkl
- filtered_avalanche_times_full.pkl
